In [55]:
import pandas as pd
import re
from pathlib import Path

# Define project paths
BASE_DIR = Path(".").resolve().parent

RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Ensure directories exist
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", BASE_DIR)

Working directory: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect


In [ ]:
# Load datasets
df = pd.read_csv(RAW_DIR / "animes.csv")
df2_path = RAW_DIR / "ratings.csv"  # DO NOT LOAD FULL
#df2 = pd.read_csv(RAW_DIR / "ratings.csv", nrows=100_000)  # Load only a sample for initial exploration
df3 = pd.read_csv(RAW_DIR / "animes2.csv")

print("Dataset 1 shape:", df.shape)
print("Dataset 2 (ratings): LARGE FILE")
print("Dataset 3 shape:", df3.shape)

Dataset 1 shape: (20237, 12)
Dataset 2 (ratings): LARGE FILE
Dataset 3 shape: (29375, 27)


In [ ]:
# Strip whitespace from column names
df.columns = df.columns.str.strip()
df3.columns = df3.columns.str.strip()

In [ ]:
# Extract MAL ID from URL
def extract_mal_id(url):
    if pd.isna(url):
        return None
    match = re.search(r"/anime/(\d+)", str(url))
    return int(match.group(1)) if match else None

df["mal_id"] = df["mal_url"].apply(extract_mal_id)

# Check results
print(df[["animeID", "mal_url", "mal_id"]].head())

   animeID                              mal_url  mal_id
0        1    https://myanimelist.net/anime/431     431
1        2   https://myanimelist.net/anime/1535    1535
2        3  https://myanimelist.net/anime/15315   15315
3        4  https://myanimelist.net/anime/14345   14345
4        5  https://myanimelist.net/anime/11757   11757


In [ ]:
print("Total rows:", len(df))
print("Valid MAL IDs:", df["mal_id"].notna().sum())
print("Missing MAL IDs:", df["mal_id"].isna().sum())

Total rows: 20237
Valid MAL IDs: 20237
Missing MAL IDs: 0


In [ ]:
# Map old animeID to new mal_id
df.rename(columns={"animeID": "old_animeID"}, inplace=True)
df.rename(columns={"mal_id": "animeID"}, inplace=True)

# Create mapping
id_map = df[["old_animeID", "animeID"]].dropna()

# Save mapping (important for reproducibility)
id_map.to_csv(INTERIM_DIR / "anime_id_map.csv", index=False)

# Convert to dictionary
id_dict = dict(zip(id_map["old_animeID"], id_map["animeID"]))

In [63]:
# Process ratings in chunks (So RAM is not overwhelmed...)
chunksize = 2_000_000
output_file = PROCESSED_DIR / "ratings_processed.csv"

first_chunk = True

for chunk in pd.read_csv(df2_path, chunksize=chunksize):
    
    # Map IDs
    chunk["animeID"] = chunk["animeID"].map(id_dict)
    
    # Filter valid rows (NO full copy)
    chunk = chunk[chunk["animeID"].notna()]
    
    # Optimize memory
    chunk["animeID"] = chunk["animeID"].astype("int32")
    chunk["userID"] = chunk["userID"].astype("int32")
    
    # Save incrementally
    chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )
    
    first_chunk = False

print("Ratings processed and saved.")

✅ Ratings processed and saved.


In [ ]:
# Select only usefull columns from dataset 1
df_subset = df[[
    "animeID",
    "sequel",
    "mal_url",
    "genres_detailed"
]].copy()

# Align dataset 3 key
df3.rename(columns={"id": "animeID"}, inplace=True)

# Merge
df_final = df3.merge(df_subset, on="animeID", how="left")

In [65]:
print("Total rows:", len(df_final))
print("Rows with extra features:", df_final["sequel"].notna().sum())

df_final[["animeID", "title", "sequel"]].head()

Total rows: 29375
Rows with extra features: 20064


,animeID,title,sequel
0,1,Cowboy Bebop,False
1,5,Cowboy Bebop: Tengoku no Tobira,False
2,6,Trigun,False
3,7,Witch Hunter Robin,False
4,8,Bouken Ou Beet,False


In [66]:
df_final.to_csv(PROCESSED_DIR / "anime_merged.csv", index=False)

print("Catalog dataset saved.")

Catalog dataset saved.


In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29375 entries, 0 to 29374
Data columns (total 30 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   animeID                           29375 non-null  int64  
 1   title                             29375 non-null  object 
 2   picture                           29146 non-null  object 
 3   title_en                          12848 non-null  object 
 4   title_ja                          29198 non-null  object 
 5   start_date                        28504 non-null  object 
 6   end_date                          26389 non-null  object 
 7   synopsis                          24240 non-null  object 
 8   mean_score                        18947 non-null  float64
 9   rank                              22326 non-null  float64
 10  popularity                        29375 non-null  int64  
 11  nsfw                              29375 non-null  bool   
 12  medi

In [ ]:
# finding NaN values
print(df_final.isnull().sum())
print(df_final.columns.to_list())

animeID                                 0
title                                   0
picture                               229
title_en                            16527
title_ja                              177
start_date                            871
end_date                             2986
synopsis                             5135
mean_score                          10428
rank                                 7049
popularity                              0
nsfw                                    0
media_type                              0
status                                  0
genres                                  0
num_episodes                            0
source                               2772
average_episode_duration_seconds        0
rating                                604
studios                             11948
num_list_users                          0
num_scoring_users                       0
watching                                0
completed                         

In [ ]:
# finding NaN values
print(df_final.isnull().sum())
print(df_final.columns.to_list())